In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

print("Ready!")

c:\Users\devuser3\AppData\Roaming\uv\python\cpython-3.14.3-windows-x86_64-none\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3083.45it/s]


Ready!


In [2]:
text = "The cat sat on the"
inputs = tokenizer(text, return_tensors="pt")

# Normal output
with torch.no_grad():
    normal_out = model(**inputs)
normal_probs = torch.softmax(normal_out.logits[0, -1, :], dim=0)
top5_normal  = torch.topk(normal_probs, 5)

print(f"Normal top 5 predictions:")
for prob, idx in zip(top5_normal.values, top5_normal.indices):
    print(f"  '{tokenizer.decode([idx])}': {prob:.4f}")

# Specific neurons 
def lesion_neuron(layer_idx, neuron_idx):
    def hook(module, input, output):
        modified = output.clone()
        modified[:, :, neuron_idx] = 0
        return modified
    return model.transformer.h[layer_idx].mlp.act.register_forward_hook(hook)

test_neurons = [
    (0,  100), (0,  500),
    (5,  200), (5,  1000),
    (11, 300), (11, 2000),
]

print(f"\nLesioning Results:")
print("-"*55)

for layer_idx, neuron_idx in test_neurons:
    hook = lesion_neuron(layer_idx, neuron_idx)
    
    with torch.no_grad():
        lesioned_out = model(**inputs)
    
    hook.remove()
    
    lesioned_probs = torch.softmax(
        lesioned_out.logits[0, -1, :], dim=0)
    top1_word = tokenizer.decode(
        [lesioned_probs.argmax().item()])
    top1_prob  = lesioned_probs.max().item()
    
    changed = "⚠️" if top1_word != " floor" else "✅"
    print(f"L{layer_idx} N{neuron_idx:<6} → "
          f"'{top1_word}' ({top1_prob:.4f}) {changed}")

Normal top 5 predictions:
  ' floor': 0.0764
  ' bed': 0.0653
  ' couch': 0.0541
  ' ground': 0.0521
  ' edge': 0.0478

Lesioning Results:
-------------------------------------------------------
L0 N100    → ' floor' (0.0768) ✅
L0 N500    → ' floor' (0.0768) ✅
L5 N200    → ' floor' (0.0769) ✅
L5 N1000   → ' floor' (0.0764) ✅
L11 N300    → ' floor' (0.0761) ✅
L11 N2000   → ' floor' (0.0762) ✅


In [3]:
def lesion_multiple_neurons(layer_idx, neuron_indices):
    def hook(module, input, output):
        modified = output.clone()
        for n in neuron_indices:
            modified[:, :, n] = 0
        return modified
    return model.transformer.h[layer_idx].mlp.act.register_forward_hook(hook)

print("Multiple neuron lesioning:")
print("-"*55)

layer_idx  = 11
n_neurons  = [1, 10, 50, 100, 500, 1000, 2000, 3000]

for n in n_neurons:
    
    np.random.seed(42)
    neurons = np.random.choice(3072, n, replace=False)
    
    hook = lesion_multiple_neurons(layer_idx, neurons)
    
    with torch.no_grad():
        lesioned_out = model(**inputs)
    
    hook.remove()
    
    lesioned_probs = torch.softmax(
        lesioned_out.logits[0, -1, :], dim=0)
    top1_word = tokenizer.decode([lesioned_probs.argmax().item()])
    top1_prob  = lesioned_probs.max().item()
    
    changed = "⚠️" if top1_word != " floor" else "✅"
    print(f"N={n:<6} → '{top1_word}' ({top1_prob:.4f}) {changed}")

Multiple neuron lesioning:
-------------------------------------------------------
N=1      → ' floor' (0.0763) ✅
N=10     → ' floor' (0.0697) ✅
N=50     → ' floor' (0.0677) ✅
N=100    → ' floor' (0.0772) ✅
N=500    → ' floor' (0.0988) ✅
N=1000   → ' floor' (0.1433) ✅
N=2000   → ' floor' (0.1158) ✅
N=3000   → ' floor' (0.1885) ✅



* Lesioning a single neuron has no effect.
* Even after lesioning 3000 neurons, the output remains the same.
* GPT-2 is highly robust to neuron lesioning.
* Some neurons act as noise — removing them can even improve results.
